# AdaDA-TransUNet — Lightning AI Setup Guide

This notebook walks through migrating from Kaggle to a **Lightning AI Studio** for running DA-TransUNet and AdaDA-TransUNet experiments.

## Why Lightning AI vs Kaggle
| Concern | Kaggle | Lightning AI |
|---|---|---|
| Filesystem | Ephemeral (`/kaggle/working/`) | Persistent (`/teamspace/studios/this_studio/`) |
| Input data | Mounted datasets (`/kaggle/input/`) | Upload once, stays forever |
| Python packages | Re-install every session | Install once, persist |
| ViT weight filename | `+` stripped → needs rename workaround | No stripping, filename used as-is |
| GPU limit | 30 h/week (free tier) | Purchased hours, no weekly cap |
| Code | Copied from Kaggle dataset each session | `git clone` once |

## Setup vs Per-Run cells
- Cells marked **[ONE-TIME SETUP]** only need to run the first time you create the studio.
- Cells marked **[PER RUN]** run each time you start a new training or test job.

Run this notebook from the **Lightning AI Studio Jupyter environment** (not locally).

---
## Step 1 — Studio  [ALREADY DONE]

Your existing studio **`deploy-model-devbox`** is ready to use. No new studio needed.

Open it at [lightning.ai](https://lightning.ai) → Studios → **deploy-model-devbox** → Start → Jupyter.

> **Path note:** Lightning AI always mounts the current studio's persistent storage at `/teamspace/studios/this_studio/` regardless of the studio name. All commands in this notebook use that fixed path and work with `deploy-model-devbox` as-is.

**GPU check** — if you want to switch to a larger GPU (e.g. A10G for AdaDA-3skip or 2-GPU DataParallel), stop the studio and change the GPU from the studio settings before starting it again.

---
## Step 2 — Clone the Repository  [ONE-TIME SETUP]

In [ ]:
%%bash
cd /teamspace/studios/this_studio
if [ ! -d AdaDA-TransUNet ]; then
    git clone https://github.com/jiaweizhong/AdaDA-TransUNet.git
    echo "Cloned successfully."
else
    echo "Repo already exists. Pulling latest changes..."
    git -C AdaDA-TransUNet pull
fi

---
## Step 3 — Install Dependencies  [ONE-TIME SETUP]

These persist in the studio — you only need to run this once.

In [ ]:
%%bash
pip install timm einops ml-collections medpy SimpleITK tensorboardX thop h5py

---
## Step 4 — Upload Data  [ONE-TIME SETUP]

You need two things:
- **Synapse dataset** — `train_npz/` and `test_vol_h5/` directories
- **ViT pretrained weights** — `R50+ViT-B_16.npz`

### Option A — Kaggle API for Synapse + Google bucket for ViT weights (recommended)

The `deepsotaai/vit-pretrained-weights` Kaggle dataset is private and returns 403 even for the owner via the API, so ViT weights are pulled directly from Google's public bucket instead (no auth needed, ~340 MB).

1. Regenerate your Kaggle API token at [kaggle.com](https://www.kaggle.com) → Account → API → **Create New Token**
2. Upload `kaggle.json` via the Lightning AI Studio file browser to `/teamspace/studios/this_studio/`
3. Run the cell below

In [ ]:
%%bash
# --- Option A: Kaggle API for Synapse + Google bucket for ViT weights ---
pip install -q kaggle
mkdir -p ~/.kaggle
cp /teamspace/studios/this_studio/kaggle.json ~/.kaggle/kaggle.json
chmod 600 ~/.kaggle/kaggle.json

BASE=/teamspace/studios/this_studio

# --- 1. Synapse dataset (dogcdt/synapse) ---
mkdir -p $BASE/data/raw_synapse
kaggle datasets download -d dogcdt/synapse \
    -p $BASE/data/raw_synapse --unzip
mkdir -p $BASE/data/Synapse
mv $BASE/data/raw_synapse/Synapse/train_npz $BASE/data/Synapse/train_npz
mv $BASE/data/raw_synapse/Synapse/test_vol_h5 $BASE/data/Synapse/test_vol_h5
rm -rf $BASE/data/raw_synapse
echo "Synapse data ready."

# --- 2. ViT pretrained weights — Google public bucket (no auth, always works) ---
# The Kaggle dataset deepsotaai/vit-pretrained-weights returns 403 even for the owner
# because the dataset is private; the Google bucket is the canonical public source.
mkdir -p $BASE/model/vit_checkpoint/imagenet21k
wget -q --show-progress \
    -O $BASE/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz \
    https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz
echo "ViT weights ready:"
ls -lh $BASE/model/vit_checkpoint/imagenet21k/

### Option B — Download ViT weights directly from Google (always works, no Kaggle needed)

In [ ]:
%%bash
# --- Option B: ViT weights from Google public bucket ---
mkdir -p /teamspace/studios/this_studio/model/vit_checkpoint/imagenet21k
wget -q --show-progress \
    -O /teamspace/studios/this_studio/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz \
    https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz
echo "ViT weights downloaded."

### Option C — Upload from your local machine

Use the Lightning AI Studio file browser (left sidebar) to drag and drop:
- `train_npz/` → `/teamspace/studios/this_studio/data/Synapse/train_npz/`
- `test_vol_h5/` → `/teamspace/studios/this_studio/data/Synapse/test_vol_h5/`
- `R50+ViT-B_16.npz` → `/teamspace/studios/this_studio/model/vit_checkpoint/imagenet21k/R50+ViT-B_16.npz`

> **Note:** Unlike Kaggle, Lightning AI does **not** strip the `+` from the filename. Keep the original `R50+ViT-B_16.npz` — no rename needed.

---
## Step 5 — Verify Data Layout  [ONE-TIME SETUP]

In [ ]:
%%bash
BASE=/teamspace/studios/this_studio

echo "=== ViT weights ==="
ls -lh $BASE/model/vit_checkpoint/imagenet21k/

echo ""
echo "=== Synapse train_npz (first 5 files) ==="
ls $BASE/data/Synapse/train_npz/ | head -5
echo "Total: $(ls $BASE/data/Synapse/train_npz/ | wc -l) files"

echo ""
echo "=== Synapse test_vol_h5 (first 5 files) ==="
ls $BASE/data/Synapse/test_vol_h5/ | head -5
echo "Total: $(ls $BASE/data/Synapse/test_vol_h5/ | wc -l) files"

Expected output:
```
=== ViT weights ===
-rw-r--r-- 1 user group 343M ... R50+ViT-B_16.npz    ← + sign must be present

=== Synapse train_npz (first 5 files) ===
case0005_slice000.npz
...
Total: 2211 files

=== Synapse test_vol_h5 (first 5 files) ===
case0001.npy.h5
...
Total: 12 files
```

---
## Step 6 — One-Time Code Fixes  [ONE-TIME SETUP]

Fix the HuggingFace `datasets` library shadowing issue (same as Kaggle).

In [ ]:
%%bash
BASE=/teamspace/studios/this_studio/AdaDA-TransUNet/experiments
touch $BASE/DA-TransUNet/datasets/__init__.py
touch $BASE/Ada-DA-TransUNet/datasets/__init__.py
echo "datasets/__init__.py created in both experiment dirs."

---
## Step 7 — Path Note  [ONE-TIME READ]

Both `train.py` and `test.py` resolve data and model paths **relative to the script location**:

```
experiments/DA-TransUNet/train.py  →  ../data/Synapse/   →  experiments/data/Synapse/
                                   →  ../model/          →  experiments/model/
```

On Lightning AI the repo lives at `/teamspace/studios/this_studio/AdaDA-TransUNet/`, so these paths resolve to:

```
/teamspace/studios/this_studio/AdaDA-TransUNet/experiments/data/Synapse/
/teamspace/studios/this_studio/AdaDA-TransUNet/experiments/model/
```

Create symlinks from there to where the data actually lives (run once):

In [ ]:
%%bash
REPO=/teamspace/studios/this_studio/AdaDA-TransUNet/experiments
DATA=/teamspace/studios/this_studio/data
MODEL=/teamspace/studios/this_studio/model

# Symlink data
mkdir -p $REPO/data
ln -sfn $DATA/Synapse $REPO/data/Synapse

# Symlink model weights
ln -sfn $MODEL $REPO/model

echo "Symlinks created:"
ls -la $REPO/data/
ls -la $REPO/model

---
## Step 8 — Verify GPU  [PER RUN]

In [ ]:
%%bash
nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

---
## Step 9 — Run DA-TransUNet Training  [PER RUN]

Reproduces the DA-TransUNet baseline (2-skip effective behavior, 150 epochs).

---
## Step 9a — Run in Background (close browser safely)  [ALTERNATIVE]

Run these cells instead of Steps 9–12 when you want to close the browser. Each cell writes a shell script and launches it detached with `nohup` — the training process survives even if the Jupyter kernel is killed.

### DA-TransUNet — train → test

In [ ]:
%%writefile /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet/run_da.sh
python -u train.py --dataset Synapse --vit_name R50-ViT-B_16 --max_epochs 300 --batch_size 24 --n_gpu 1 --base_lr 0.01 --n_skip 3 --img_size 224 --seed 1234 --val_interval 10 && python -u test.py --dataset Synapse --vit_name R50-ViT-B_16 --num_classes 9 --img_size 224 --is_savenii

Now launch it detached:

### DA-TransUNet — launch

In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet
mkdir -p datasets && touch datasets/__init__.py
nohup bash run_da.sh > run_da.log 2>&1 &
echo "DA PID: "


In [ ]:
%%writefile /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet/run_adada.sh
# CUDA_VISIBLE_DEVICES restricts to 2 GPUs on a 4-GPU instance for T4x2 paper experiment
CUDA_VISIBLE_DEVICES=0,1 python -u train.py --dataset Synapse --vit_name R50-ViT-B_16 --max_epochs 300 --batch_size 12 --n_gpu 2 --base_lr 0.01 --n_skip 3 --img_size 224 --seed 1234 --val_interval 10 && python -u test.py --dataset Synapse --vit_name R50-ViT-B_16 --num_classes 9 --img_size 224 --is_savenii


Now launch it detached:

### AdaDA-TransUNet — launch


In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet
mkdir -p datasets && touch datasets/__init__.py
nohup bash run_adada.sh > run_adada.log 2>&1 &
echo "AdaDA PID: "


### When you return — check progress

In [ ]:
%%bash
# Is training still running?
ps aux | grep train.py | grep -v grep

# Last 20 lines of DA-TransUNet log
echo "=== DA-TransUNet log (tail) ==="
tail -20 /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet/run_da.log

# Last 20 lines of AdaDA-TransUNet log
echo "=== AdaDA-TransUNet log (tail) ==="
tail -20 /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet/run_adada.log

# Final results
echo "=== DA Final results ==="
grep "Testing performance" /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet/run_da.log
echo "=== AdaDA Final results ==="
grep "Testing performance" /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet/run_adada.log


In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet
mkdir -p datasets && touch datasets/__init__.py
python -u train.py 
    --dataset Synapse 
    --vit_name R50-ViT-B_16 
    --max_epochs 300 
    --batch_size 24 
    --n_gpu 1 
    --base_lr 0.01 
    --n_skip 3 
    --img_size 224 
    --seed 1234 
    --val_interval 10


---
## Step 10 — Run DA-TransUNet Testing  [PER RUN]

In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/DA-TransUNet
mkdir -p datasets && touch datasets/__init__.py
python -u test.py \
    --dataset Synapse \
    --vit_name R50-ViT-B_16 \
    --num_classes 9 \
    --img_size 224 \
    --is_savenii

---
## Step 11 — Run AdaDA-TransUNet Training  [PER RUN]


In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet
mkdir -p datasets && touch datasets/__init__.py
python -u train.py 
    --dataset Synapse 
    --vit_name R50-ViT-B_16 
    --max_epochs 300 
    --batch_size 24 
    --base_lr 0.01 
    --n_skip 3 
    --img_size 224 
    --window_size 7 
    --rank 32 
    --groups 8 
    --seed 1234 
    --val_interval 10


---
## Step 12 — Run AdaDA-TransUNet Testing  [PER RUN]

In [ ]:
%%bash
cd /teamspace/studios/this_studio/AdaDA-TransUNet/experiments/Ada-DA-TransUNet
python -u test.py \
    --dataset Synapse \
    --vit_name R50-ViT-B_16 \
    --num_classes 9 \
    --img_size 224 \
    --window_size 7 \
    --rank 32 \
    --groups 8 \
    --is_savenii

---
## Checkpoint & Results Locations

All outputs persist between studio sessions — no need to re-download.

| Artifact | Path |
|---|---|
| Best checkpoint | `experiments/model/TU_Synapse224/TU/TU_pretrain_R50-ViT-B_16_skip3_epo150_bs24_224/best_model.pth` |
| Last epoch checkpoint | `...epoch_149.pth` |
| Test logs | `experiments/DA-TransUNet/test_log/` |
| Training logs | `experiments/DA-TransUNet/` (stdout captured by Jupyter cell) |
| Predictions (NIfTI) | `experiments/predictions/` (when `--is_savenii` is set) |

To download results to your local machine, use the Lightning AI Studio file browser or run:
```bash
# From your LOCAL terminal (not in this notebook):
scp -r <studio-ssh-address>:/teamspace/studios/this_studio/AdaDA-TransUNet/experiments/model ./
```
(Find the SSH address in the Lightning AI Studio settings page.)

---
## Multi-GPU Note (AdaDA-TransUNet only)

AdaDA-TransUNet supports multi-GPU  training; DA-TransUNet OOMs because its global (N^2)$ PAM must store a $ attention matrix per sample at the 112×112 skip connection.

**Running on a 4×T4 instance but want only 2 GPUs** (for a like-for-like T4×2 paper experiment):



This is already set in  above. From PyTorch’s perspective  returns 2, DataParallel wraps exactly those two T4s, and GPUs 2–3 are invisible.

| Config | Per-GPU batch | Total batch | LR | Notes |
|---|---|---|---|---|
| DA-TransUNet T4×1 | 24 | 24 | 0.01 | baseline |
| AdaDA T4×2 () | 12 | 24 | 0.01 | fair comparison |

DA-TransUNet must remain on a **single GPU** — attempting 2-GPU DataParallel will OOM during the backward pass through PAM.
